# Pilot notebook: manually sail a mission, and ask your OOW / VHF agents what to do

Two agents, two different jobs:
- **OOW agent** — reads the current navigational situation and recommends a
  manoeuvre (`turn_left`, `turn_right`, `hold_course`, `speed_up`,
  `slow_down`, `stop`), grounded in COLREG.
- **VHF agent** — given the OOW agent's chosen manoeuvre, drafts the actual
  radio call to coordinate it with the target vessel (SMCP-style phrasing).

This notebook works in two modes:
- **DEMO mode** (`LIVE_MOOS = False`, the default) — uses a canned state in
  your own `bridge_snapshot` JSON shape (same schema you've been pasting me
  all along) so every cell below runs and is inspectable with no MOOS-IvP
  install needed. Good for testing the notebook itself, prompt-engineering
  the agents, or replaying a saved snapshot you're debugging.
- **LIVE mode** (`LIVE_MOOS = True`) — connects to a running MOOS-IvP
  mission via `pymoos`, exactly like `colreg_llm_bridge_paused.py`, and
  drives the real vehicle. Point it at any of the 12 missions in
  `moos_colreg_eval_missions.zip`, or your own.

You control the vessel manually with `turn_left(deg)`, `turn_right(deg)`,
`set_speed(x)`, `stop_vessel()`, `resume()` — and separately query either
agent at any point with `ask_oow()` / `ask_vhf()` to compare what *you*
did against what the agent *would* recommend. That comparison is the whole
point of this notebook: a place to manually probe disagreements between
your own judgement and the agent's, one decision at a time.


## 1. Setup

In [ ]:
import json, math, time, sys
from dataclasses import dataclass, field

LIVE_MOOS = False   # flip to True once you point this at a real mission

if LIVE_MOOS:
    import pymoos  # pip install pymoos --break-system-packages  (Linux/cloud only, see earlier notes)

try:
    import anthropic
    _ANTHROPIC_OK = True
except ImportError:
    _ANTHROPIC_OK = False
    print("anthropic not installed (pip install anthropic) -- agent calls will use a stub response.")


## 2. Geometry + narration helpers

Same functions as `colreg_llm_bridge_paused.py` / `narrate.py` from earlier
in this project -- copied in here so this notebook is self-contained. If
you're already running those as a package, delete this cell and `import`
them instead of duplicating.


In [ ]:
def bearing_and_range(ox, oy, tx, ty):
    dx, dy = tx - ox, ty - oy
    return math.degrees(math.atan2(dx, dy)) % 360.0, math.hypot(dx, dy)

def relative_bearing(own_heading, true_bearing):
    return (true_bearing - own_heading + 540) % 360 - 180

def cpa_tcpa(ox, oy, ohdg, ospd, tx, ty, thdg, tspd):
    oh, th = math.radians(ohdg), math.radians(thdg)
    vox, voy = ospd*math.sin(oh), ospd*math.cos(oh)
    vtx, vty = tspd*math.sin(th), tspd*math.cos(th)
    dx, dy = tx-ox, ty-oy
    dvx, dvy = vtx-vox, vty-voy
    rel_sq = dvx**2 + dvy**2
    if rel_sq < 1e-6:
        return math.hypot(dx, dy), 0.0
    t = max(0.0, -(dx*dvx+dy*dvy)/rel_sq)
    return math.hypot(dx+dvx*t, dy+dvy*t), t

def classify_encounter(own_x, own_y, own_hdg, tgt_x, tgt_y, tgt_hdg):
    '''Correct Rule 13 check: overtaking is defined by the bearing of OWN-SHIP
    as seen from the TARGET (>112.5 deg abaft the target's beam), not by the
    bearing of the target as seen from own-ship. Using only the latter (an
    earlier version of this helper did exactly that) misclassifies real
    overtaking cases as ordinary crossing whenever the closing angle is
    fine/moderate rather than near-dead-astern -- confirmed against a real
    logged state where the old check said "crossing_target_on_starboard" but
    the correct Rule 13 check (own-ship 156.6 deg abaft the target's beam)
    says overtaking.'''
    brg_own_to_tgt, _ = bearing_and_range(own_x, own_y, tgt_x, tgt_y)
    rel_from_own = relative_bearing(own_hdg, brg_own_to_tgt)

    brg_tgt_to_own, _ = bearing_and_range(tgt_x, tgt_y, own_x, own_y)
    rel_from_tgt = relative_bearing(tgt_hdg, brg_tgt_to_own)

    course_diff = (tgt_hdg - own_hdg + 540) % 360 - 180
    if abs(rel_from_own) <= 6 and abs(abs(course_diff) - 180) <= 20:
        return "head_on", ["Rule 14"], rel_from_own
    if abs(rel_from_tgt) > 112.5:
        return "we_are_overtaking_target", ["Rule 13"], rel_from_own
    if abs(rel_from_own) > 112.5:
        return "target_is_overtaking_us", ["Rule 13"], rel_from_own
    if rel_from_own > 0:
        return "crossing_target_on_starboard", ["Rule 15", "Rule 16"], rel_from_own
    return "crossing_target_on_port", ["Rule 15", "Rule 17"], rel_from_own

def narrate(state):
    '''state: dict in your bridge_snapshot['state'] shape (own_ship, contacts, mission, ...).'''
    own = state["own_ship"]
    lines = [f"Own-ship {own.get('name','LLM_SHIP')} at ({own['x']:.1f}, {own['y']:.1f}), "
             f"heading {own['heading']:.1f}, speed {own['speed']:.2f}."]
    mission = state.get("mission")
    if mission:
        lines.append(f"Mission waypoint at ({mission['x']:.1f}, {mission['y']:.1f}).")
    contacts = state.get("contacts", [])
    if not contacts:
        lines.append("No contacts tracked.")
    else:
        lines.append(f"{len(contacts)} contact(s):")
        for c in contacts:
            _, rng = bearing_and_range(own["x"], own["y"], c["x"], c["y"])
            cpa, tcpa = cpa_tcpa(own["x"], own["y"], own["heading"], own["speed"],
                                  c["x"], c["y"], c["heading"], c["speed"])
            enc, rules, rel = classify_encounter(own["x"], own["y"], own["heading"],
                                                  c["x"], c["y"], c["heading"])
            lines.append(f"  - {c['name']}: range {rng:.0f}m, rel.bearing {rel:.1f} deg, "
                         f"heading {c['heading']:.1f}, speed {c['speed']:.2f}, "
                         f"CPA {cpa:.0f}m, TCPA {tcpa:.0f}s, encounter={enc}, rules={rules}")
    return "\n".join(lines)


## 3. Demo state (edit freely, or load a real `bridge_snapshot` you've saved)

This is your own MOOS snapshot format from earlier in the project --
paste any real one you want to replay here instead.


In [ ]:
DEMO_STATE = {
    "own_ship": {"name": "LLM_SHIP", "x": -292.6, "y": -336.4, "heading": 48.1, "speed": 17.0},
    "mission": {"x": 452.6, "y": 325.6},
    "contacts": [
        {"name": "RANDOM_TS3", "x": 103.8, "y": -224.8, "heading": 50.9, "speed": 12.0},
    ],
}


## 4. MOOS connection (LIVE mode only)

Mirrors `OwnShipLink` + `MultiVehiclePause` from `colreg_llm_bridge_paused.py`.
If `LIVE_MOOS = False` this whole cell is skipped and manual commands /
`step()` just operate on `DEMO_STATE` in memory instead -- see cell 6.


In [ ]:
class LiveLink:
    def __init__(self, name, host, port, other_vehicles=()):
        self.name = name
        self.own = {"x":0.0,"y":0.0,"heading":0.0,"speed":0.0}
        self.contacts_raw = {}
        self.comms = pymoos.comms()
        self.comms.set_on_connect_callback(self._on_connect)
        self.comms.set_on_mail_callback(self._on_mail)
        self.comms.run(host, port, f"{name}_notebook")
        self.pausers = [self._make_pauser(n,h,p) for n,h,p in ((name,host,port),)+tuple(other_vehicles)]

    def _make_pauser(self, name, host, port):
        c = pymoos.comms()
        c.set_on_connect_callback(lambda: True)
        c.run(host, port, f"pauser_{name}")
        time.sleep(0.2)
        return c

    def _on_connect(self):
        for v in ("NAV_X","NAV_Y","NAV_HEADING","NAV_SPEED"):
            self.comms.register(v, 0)
        self.comms.register("NODE_REPORT", 0)
        return True

    def _on_mail(self):
        for msg in self.comms.fetch():
            k = msg.key()
            if k=="NAV_X": self.own["x"]=msg.double()
            elif k=="NAV_Y": self.own["y"]=msg.double()
            elif k=="NAV_HEADING": self.own["heading"]=msg.double()
            elif k=="NAV_SPEED": self.own["speed"]=msg.double()
            elif k=="NODE_REPORT": self._parse(msg.string())
        return True

    def _parse(self, s):
        f = dict(kv.split("=",1) for kv in s.split(",") if "=" in kv)
        name = f.get("NAME")
        if not name or name==self.name: return
        try:
            self.contacts_raw[name] = {"x":float(f.get("X",0)),"y":float(f.get("Y",0)),
                                        "heading":float(f.get("HDG",f.get("HEADING",0))),
                                        "speed":float(f.get("SPD",f.get("SPEED",0)))}
        except ValueError:
            pass

    def snapshot(self):
        return {"own_ship": dict(self.own, name=self.name),
                "mission": None,
                "contacts": [dict(v, name=k) for k,v in self.contacts_raw.items()]}

    def publish(self, heading, speed):
        now = pymoos.time()
        self.comms.notify("DESIRED_HEADING", float(heading), now)
        self.comms.notify("DESIRED_SPEED", float(speed), now)

    def pause_all(self, paused=True):
        for p in self.pausers:
            p.notify("USM_SIM_PAUSED", "true" if paused else "false", pymoos.time())

link = None
if LIVE_MOOS:
    # edit to match your mission's ports, e.g. from moos_colreg_eval_missions.zip
    link = LiveLink("opship", "localhost", 9001, other_vehicles=[("ts1","localhost",9002)])
    time.sleep(2.0)
    print("Connected. link.snapshot() ->", link.snapshot())


## 5. Manual controls — Left / Right / Stop / Faster / Slower

These are what you asked for directly. In DEMO mode they mutate
`DEMO_STATE` in place so you can see the effect immediately; in LIVE mode
they publish to MOOS (pausing/resuming the sim around the command, same
pattern as the earlier pause-aware bridge, so nothing races while you
think).


In [ ]:
def _current_state():
    return link.snapshot() if LIVE_MOOS else DEMO_STATE

def _apply(new_heading=None, new_speed=None):
    if LIVE_MOOS:
        link.pause_all(True)
        own = link.snapshot()["own_ship"]
        h = own["heading"] if new_heading is None else new_heading
        s = own["speed"] if new_speed is None else new_speed
        link.publish(h, s)
        link.pause_all(False)
    else:
        if new_heading is not None:
            DEMO_STATE["own_ship"]["heading"] = new_heading % 360.0
        if new_speed is not None:
            DEMO_STATE["own_ship"]["speed"] = max(0.0, new_speed)

def turn_left(degrees=10):
    own = _current_state()["own_ship"]
    _apply(new_heading=own["heading"] - degrees)
    print(f"Turned {degrees} deg to port. New heading: {_current_state()['own_ship']['heading']:.1f}")

def turn_right(degrees=10):
    own = _current_state()["own_ship"]
    _apply(new_heading=own["heading"] + degrees)
    print(f"Turned {degrees} deg to starboard. New heading: {_current_state()['own_ship']['heading']:.1f}")

def set_speed(new_speed):
    _apply(new_speed=new_speed)
    print(f"Speed set to {new_speed}. ")

def speed_up(delta=2.0):
    own = _current_state()["own_ship"]
    set_speed(own["speed"] + delta)

def slow_down(delta=2.0):
    own = _current_state()["own_ship"]
    set_speed(max(0.0, own["speed"] - delta))

def stop_vessel():
    set_speed(0.0)
    print("Stopped.")

def hold_course():
    print(f"Holding course {_current_state()['own_ship']['heading']:.1f} / "
          f"speed {_current_state()['own_ship']['speed']:.2f}.")

def step(seconds=5.0):
    '''Advance simulated time. LIVE mode: unpause, sleep, repause (see
    PAUSE_MODE_README.md for why). DEMO mode: dead-reckon every vessel
    forward by `seconds` at current heading/speed.'''
    if LIVE_MOOS:
        link.pause_all(False)
        time.sleep(seconds)
        link.pause_all(True)
    else:
        def advance(v, dt):
            h = math.radians(v["heading"])
            v["x"] += v["speed"]*math.sin(h)*dt
            v["y"] += v["speed"]*math.cos(h)*dt
        advance(DEMO_STATE["own_ship"], seconds)
        for c in DEMO_STATE["contacts"]:
            advance(c, seconds)
    print(narrate(_current_state()))


## 6. The two agents

`ask_oow()` recommends ONE of: `turn_left`, `turn_right`, `hold_course`,
`speed_up`, `slow_down`, `stop` (+ a magnitude where relevant), grounded
in COLREG. `ask_vhf()` takes that decision and drafts the actual radio
call, in the style of your VHF gold-answer dataset (SMCP phrasing,
prowords, correct channel handling).

Both fall back to an obviously-labelled stub if `anthropic` isn't
installed / no API key is set, so this notebook stays runnable without
credentials while you're just testing the plumbing.


In [ ]:
OOW_SYSTEM_PROMPT = """You are the Officer of the Watch reasoning system for \
an autonomous vessel. Given the situation described, recommend exactly ONE \
manoeuvre, grounded in COLREG. Reply with ONLY a JSON object:
{"action": "turn_left|turn_right|hold_course|speed_up|slow_down|stop",
 "degrees": <float, only for turn_left/turn_right>,
 "rule_applied": "<e.g. Rule 15>",
 "reasoning": "<one or two sentences>"}
Rules of thumb: give-way vessels alter early and substantially, normally to \
starboard; stand-on vessels hold course/speed unless the other vessel \
clearly isn't keeping clear; head-on situations mean both vessels alter to \
starboard; never recommend altering to port toward a vessel that is itself \
on your port side while you are avoiding collision."""

VHF_SYSTEM_PROMPT = """You are the VHF radio operator for this vessel. Given \
the situation and the manoeuvre the OOW has decided on, draft the actual \
radio call to coordinate with the target vessel, in correct SMCP / prowords \
style (THIS IS, OVER, correct channel handling: hail on 16, propose a \
working channel). Reply with ONLY a JSON object:
{"channel_hailing": "16", "channel_working": "<e.g. 13>",
 "transmission": "<the exact words to transmit, prowords included>"}"""

def _call_llm(system, user_prompt, model="claude-sonnet-4-6"):
    if not _ANTHROPIC_OK:
        return {"_stub": True, "note": "anthropic not installed -- this is a placeholder, not a real agent response."}
    try:
        client = anthropic.Anthropic()
        resp = client.messages.create(model=model, max_tokens=400, system=system,
                                       messages=[{"role":"user","content":user_prompt}])
        text = "".join(b.text for b in resp.content if getattr(b,"type",None)=="text").strip()
        import re
        text = re.sub(r"^```(json)?|```$", "", text, flags=re.MULTILINE).strip()
        return json.loads(text)
    except Exception as e:
        return {"_stub": True, "note": f"LLM call failed ({e}) -- placeholder response."}

def ask_oow():
    state = _current_state()
    prompt = narrate(state) + "\n\nReturn your recommended manoeuvre as the specified JSON object."
    decision = _call_llm(OOW_SYSTEM_PROMPT, prompt)
    print("OOW agent recommends:", json.dumps(decision, indent=2))
    return decision

def ask_vhf(oow_decision=None):
    state = _current_state()
    if oow_decision is None:
        oow_decision = ask_oow()
    prompt = (narrate(state) + "\n\nOOW decision: " + json.dumps(oow_decision) +
              "\n\nDraft the radio call for this manoeuvre as the specified JSON object.")
    call = _call_llm(VHF_SYSTEM_PROMPT, prompt)
    print("VHF agent drafts:", json.dumps(call, indent=2))
    return call


## 7. Try it

Run these cells interactively -- pilot manually, or ask the agents, or
both and compare. This is the actual workflow: look at `narrate()`'s
output, decide what *you* would do, THEN call `ask_oow()` and see if it
agrees. Disagreements are the interesting part.


In [ ]:
print(narrate(_current_state()))


In [ ]:
# Manual piloting, e.g.:
# turn_right(20)
# step(10)
# print(narrate(_current_state()))


In [ ]:
# Ask the agents what they'd do from the CURRENT state:
oow_decision = ask_oow()
vhf_call = ask_vhf(oow_decision)


In [ ]:
# If you agree with the OOW agent, execute its decision manually and step forward:
# action = oow_decision.get("action")
# if action == "turn_left": turn_left(oow_decision.get("degrees", 10))
# elif action == "turn_right": turn_right(oow_decision.get("degrees", 10))
# elif action == "speed_up": speed_up()
# elif action == "slow_down": slow_down()
# elif action == "stop": stop_vessel()
# else: hold_course()
# step(10)


## 8. Loading a real Imazu-style mission (LIVE mode)

Point `LiveLink` at any scenario folder from `moos_colreg_eval_missions.zip`.
Each scenario's own generated `.moos` files tell you the right ports (see
that scenario's `README.md`). Launch the mission from the shell first
(`./launch.sh`), then set `LIVE_MOOS = True` above and re-run from cell 4.

Given section 1's finding (86% of real encounters need no action at all),
consider also replaying some "quiet" `DEMO_STATE`s -- very long TCPA,
large CPA -- to check the OOW agent correctly recommends `hold_course`
rather than manoeuvring unnecessarily, not just that it reacts correctly
when there IS a threat.
